# ABB Motor Predictive Maintenance — Logistic Regression

Bu notebook:

- BigQuery `training_dataset` tablosunu çeker,
- zaman bazlı TRAIN / VALIDATION / TEST ayrımını kullanır,
- Logistic Regression modeli kurar,
- validation verisiyle alarm threshold'u seçer,
- test performansını ölçer,
- risk probability ve condition score üretir,
- sonuçları `pm_ml` tablolarına yazar.

> Hedef, gerçek arıza değil; sonraki üç aktif ölçüm içinde anormal sensör davranışı görülmesidir.


## 1. Kurulum ve proje ayarları

In [ ]:
import json
import os
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.config import GCP_PROJECT_ID

TRAINING_TABLE = f"{GCP_PROJECT_ID}.pm_features.training_dataset"
FEATURE_TABLE = f"{GCP_PROJECT_ID}.pm_features.motor_condition_features"

ML_DATASET = "pm_ml"
PREDICTION_TABLE = f"{GCP_PROJECT_ID}.{ML_DATASET}.predictions"
METRICS_TABLE = f"{GCP_PROJECT_ID}.{ML_DATASET}.model_metrics"
COEFFICIENT_TABLE = f"{GCP_PROJECT_ID}.{ML_DATASET}.model_coefficients"

BQ_LOCATION = os.getenv("BQ_LOCATION", "EU")
MODEL_NAME = "logistic_regression_v1"

MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "model"
FIGURE_DIR = REPORT_DIR / "figures"

for folder in [MODEL_DIR, REPORT_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

credential_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
if credential_path and not Path(credential_path).exists():
    raise FileNotFoundError(
        f"Credential JSON bulunamadı: {credential_path}"
    )

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Project root:", PROJECT_ROOT)
print("Project ID:", GCP_PROJECT_ID)
print("Training table:", TRAINING_TABLE)
print("Prediction table:", PREDICTION_TABLE)


## 2. BigQuery tablolarını yükleme

In [ ]:
client = bigquery.Client(
    project=GCP_PROJECT_ID,
    location=BQ_LOCATION,
)

def read_bq_table(table_name: str) -> pd.DataFrame:
    query = f'''
    SELECT *
    FROM `{table_name}`
    ORDER BY Timestamp
    '''
    df = client.query(query).to_dataframe()
    df["Timestamp"] = pd.to_datetime(
        df["Timestamp"],
        utc=True,
        errors="coerce",
    )
    return df

df_training = read_bq_table(TRAINING_TABLE)
df_features = read_bq_table(FEATURE_TABLE)

print("Training:", df_training.shape)
print("Features:", df_features.shape)
display(df_training.head())


## 3. Veri ve split kontrolü

In [ ]:
required = [
    "Asset_ID",
    "Timestamp",
    "Operating_Mode",
    "Future_Event",
    "Data_Split",
]

missing_required = [
    c for c in required
    if c not in df_training.columns
]

if missing_required:
    raise ValueError(f"Eksik kolonlar: {missing_required}")

audit = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Unique assets",
        "Duplicate Asset_ID + Timestamp",
        "First timestamp",
        "Last timestamp",
    ],
    "Value": [
        len(df_training),
        len(df_training.columns),
        df_training["Asset_ID"].nunique(),
        df_training.duplicated(
            ["Asset_ID", "Timestamp"]
        ).sum(),
        df_training["Timestamp"].min(),
        df_training["Timestamp"].max(),
    ],
})

display(audit)
display(pd.crosstab(
    df_training["Data_Split"],
    df_training["Future_Event"],
    margins=True,
))

for split_name in ["TRAIN", "VALIDATION", "TEST"]:
    target = df_training.loc[
        df_training["Data_Split"] == split_name,
        "Future_Event",
    ]
    if target.empty or target.nunique() < 2:
        raise ValueError(
            f"{split_name} grubunda hem 0 hem 1 bulunmalı."
        )


In [ ]:
target_counts = (
    df_training["Future_Event"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(7, 5))
plt.bar(
    target_counts.index.astype(str),
    target_counts.values,
)
plt.title("Future Event Class Distribution")
plt.xlabel("Future event")
plt.ylabel("Row count")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "01_target_distribution.png",
    dpi=150,
)
plt.show()


## 4. Feature seçimi

Leakage yaratabilecek `Current_Anomaly`, `Anomaly_Indicator_Count` ve `Is_High_*` kolonları kullanılmaz.

`Overall_Vib_mm_s`, yönsel titreşimlerin maksimumuyla aynı bilgiyi büyük ölçüde taşıdığı için ilk modelde dışarıda bırakılır.


In [ ]:
numeric_features = [
    "Speed_rpm",
    "Output_Power_kW",
    "Skin_Temp_C",
    "Vib_Radial_mm_s",
    "Vib_Tangential_mm_s",
    "Vib_Axial_mm_s",
    "Acc_RMS_Axial_g",
    "Acc_RMS_Tangential_g",
    "Acc_RMS_Radial_g",
    "Pk_Pk_Tangential_g",
    "Hours_Since_Previous_Measurement",
    "Change_Vib_Radial",
    "Change_Vib_Tangential",
    "Change_Vib_Axial",
    "Change_Acc_Tangential",
    "Change_Skin_Temp",
    "Avg_3_Vib_Radial",
    "Avg_3_Vib_Tangential",
    "Avg_3_Vib_Axial",
    "Max_3_Vib_Tangential",
    "Max_3_Acc_Tangential",
    "Max_3_Pk_Pk_Tangential",
    "Avg_3_Skin_Temp",
    "Deviation_Vib_Radial",
    "Deviation_Vib_Tangential",
    "Deviation_Vib_Axial",
    "Deviation_Skin_Temp",
    "Ratio_Vib_Radial_To_Normal",
    "Ratio_Vib_Tangential_To_Normal",
    "Ratio_Vib_Axial_To_Normal",
]

categorical_features = ["Operating_Mode"]
feature_columns = numeric_features + categorical_features
target_column = "Future_Event"

missing_features = [
    c for c in feature_columns
    if c not in df_training.columns
]

if missing_features:
    raise ValueError(f"Eksik feature kolonları: {missing_features}")

for col in numeric_features:
    df_training[col] = pd.to_numeric(
        df_training[col],
        errors="coerce",
    )
    df_features[col] = pd.to_numeric(
        df_features[col],
        errors="coerce",
    )

for col in categorical_features:
    df_training[col] = df_training[col].astype("string")
    df_features[col] = df_features[col].astype("string")

df_training[target_column] = (
    pd.to_numeric(df_training[target_column])
    .astype(int)
)

feature_audit = pd.DataFrame({
    "Feature": feature_columns,
    "Missing_Count": [
        df_training[c].isna().sum()
        for c in feature_columns
    ],
    "Unique_Count": [
        df_training[c].nunique(dropna=True)
        for c in feature_columns
    ],
})

feature_audit["Missing_Percentage"] = (
    100
    * feature_audit["Missing_Count"]
    / len(df_training)
)

display(
    feature_audit.sort_values(
        "Missing_Percentage",
        ascending=False,
    )
)


## 5. TRAIN / VALIDATION / TEST hazırlığı

In [ ]:
train_mask = df_training["Data_Split"] == "TRAIN"
validation_mask = df_training["Data_Split"] == "VALIDATION"
test_mask = df_training["Data_Split"] == "TEST"

X_train = df_training.loc[train_mask, feature_columns].copy()
y_train = df_training.loc[train_mask, target_column].copy()

X_validation = df_training.loc[
    validation_mask,
    feature_columns,
].copy()
y_validation = df_training.loc[
    validation_mask,
    target_column,
].copy()

X_test = df_training.loc[test_mask, feature_columns].copy()
y_test = df_training.loc[test_mask, target_column].copy()

split_summary = pd.DataFrame({
    "Split": ["TRAIN", "VALIDATION", "TEST"],
    "Rows": [len(X_train), len(X_validation), len(X_test)],
    "Events": [y_train.sum(), y_validation.sum(), y_test.sum()],
    "Event_Rate": [
        y_train.mean(),
        y_validation.mean(),
        y_test.mean(),
    ],
})

display(split_summary)


## 6. Dummy baseline ve Logistic Regression pipeline

In [ ]:
dummy_model = DummyClassifier(strategy="prior")
dummy_model.fit(
    np.zeros((len(y_train), 1)),
    y_train,
)

dummy_test_probability = dummy_model.predict_proba(
    np.zeros((len(y_test), 1))
)[:, 1]

dummy_test_prediction = (
    dummy_test_probability >= 0.5
).astype(int)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore"),
    ),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    (
        "categorical",
        categorical_pipeline,
        categorical_features,
    ),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42,
        ),
    ),
])

model.fit(X_train, y_train)
print("Model eğitildi.")


## 7. Validation üzerinden threshold seçimi

In [ ]:
validation_probability = model.predict_proba(
    X_validation
)[:, 1]

precision_values, recall_values, thresholds = (
    precision_recall_curve(
        y_validation,
        validation_probability,
    )
)

threshold_results = pd.DataFrame({
    "Threshold": thresholds,
    "Precision": precision_values[:-1],
    "Recall": recall_values[:-1],
})

threshold_results["F1"] = np.where(
    (
        threshold_results["Precision"]
        + threshold_results["Recall"]
    ) > 0,
    (
        2
        * threshold_results["Precision"]
        * threshold_results["Recall"]
        / (
            threshold_results["Precision"]
            + threshold_results["Recall"]
        )
    ),
    0,
)

beta = 2

threshold_results["F2"] = np.where(
    (
        beta**2 * threshold_results["Precision"]
        + threshold_results["Recall"]
    ) > 0,
    (
        (1 + beta**2)
        * threshold_results["Precision"]
        * threshold_results["Recall"]
        / (
            beta**2 * threshold_results["Precision"]
            + threshold_results["Recall"]
        )
    ),
    0,
)

candidates = threshold_results[
    threshold_results["Threshold"].between(0.05, 0.95)
].copy()

if candidates.empty:
    candidates = threshold_results.copy()

best_row = (
    candidates
    .sort_values(
        ["F2", "Recall", "Precision"],
        ascending=False,
    )
    .iloc[0]
)

selected_threshold = float(best_row["Threshold"])

display(
    candidates
    .sort_values("F2", ascending=False)
    .head(15)
)

print("Selected threshold:", round(selected_threshold, 4))
print("Validation ROC-AUC:", round(
    roc_auc_score(
        y_validation,
        validation_probability,
    ),
    4,
))
print("Validation Average Precision:", round(
    average_precision_score(
        y_validation,
        validation_probability,
    ),
    4,
))
print("Validation Precision:", round(best_row["Precision"], 4))
print("Validation Recall:", round(best_row["Recall"], 4))
print("Validation F2:", round(best_row["F2"], 4))


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(
    threshold_results["Threshold"],
    threshold_results["Precision"],
    label="Precision",
)
plt.plot(
    threshold_results["Threshold"],
    threshold_results["Recall"],
    label="Recall",
)
plt.plot(
    threshold_results["Threshold"],
    threshold_results["F1"],
    label="F1",
)
plt.plot(
    threshold_results["Threshold"],
    threshold_results["F2"],
    label="F2",
)
plt.axvline(
    selected_threshold,
    linestyle="--",
    label="Selected threshold",
)
plt.title("Validation Metrics by Threshold")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "02_threshold_metrics.png",
    dpi=150,
)
plt.show()


## 8. Test performansı

In [ ]:
test_probability = model.predict_proba(X_test)[:, 1]
test_prediction_default = (
    test_probability >= 0.50
).astype(int)
test_prediction_selected = (
    test_probability >= selected_threshold
).astype(int)

print(
    classification_report(
        y_test,
        test_prediction_selected,
        digits=3,
        zero_division=0,
    )
)


In [ ]:
def metric_row(
    label,
    y_true,
    probability,
    prediction,
    threshold,
):
    return {
        "Model": label,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, prediction),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true,
            prediction,
        ),
        "Precision": precision_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "F1": f1_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "F2": fbeta_score(
            y_true,
            prediction,
            beta=2,
            zero_division=0,
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            probability,
        ),
        "Average_Precision": average_precision_score(
            y_true,
            probability,
        ),
    }

test_metrics = pd.DataFrame([
    metric_row(
        "Dummy baseline",
        y_test,
        dummy_test_probability,
        dummy_test_prediction,
        0.50,
    ),
    metric_row(
        "Logistic Regression — threshold 0.50",
        y_test,
        test_probability,
        test_prediction_default,
        0.50,
    ),
    metric_row(
        "Logistic Regression — selected threshold",
        y_test,
        test_probability,
        test_prediction_selected,
        selected_threshold,
    ),
])

display(test_metrics)


In [ ]:
matrix = confusion_matrix(
    y_test,
    test_prediction_selected,
)

display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=["No Event", "Future Event"],
)

display_matrix.plot(values_format="d")
plt.title("Test Confusion Matrix")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "03_test_confusion_matrix.png",
    dpi=150,
)
plt.show()


## 9. ROC, Precision–Recall ve calibration grafikleri

In [ ]:
fpr, tpr, _ = roc_curve(
    y_test,
    test_probability,
)

plt.figure(figsize=(8, 6))
plt.plot(
    fpr,
    tpr,
    label=f"AUC={roc_auc_score(y_test, test_probability):.3f}",
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random baseline",
)
plt.title("Test ROC Curve")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "04_test_roc_curve.png",
    dpi=150,
)
plt.show()


In [ ]:
test_precision, test_recall, _ = precision_recall_curve(
    y_test,
    test_probability,
)

plt.figure(figsize=(8, 6))
plt.plot(
    test_recall,
    test_precision,
    label=(
        "AP="
        f"{average_precision_score(y_test, test_probability):.3f}"
    ),
)
plt.axhline(
    y_test.mean(),
    linestyle="--",
    label="Positive class rate",
)
plt.title("Test Precision–Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "05_test_precision_recall_curve.png",
    dpi=150,
)
plt.show()


In [ ]:
fraction_positive, mean_probability = calibration_curve(
    y_test,
    test_probability,
    n_bins=8,
    strategy="quantile",
)

plt.figure(figsize=(8, 6))
plt.plot(
    mean_probability,
    fraction_positive,
    marker="o",
    label="Logistic Regression",
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration",
)
plt.title("Probability Calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed event rate")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "06_probability_calibration.png",
    dpi=150,
)
plt.show()


## 10. Katsayılar ve odds ratio

In [ ]:
feature_names = (
    model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    model
    .named_steps["classifier"]
    .coef_[0]
)

coefficient_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
})

coefficient_df["Feature"] = (
    coefficient_df["Feature"]
    .str.replace("numeric__", "", regex=False)
    .str.replace("categorical__", "", regex=False)
)

coefficient_df["Absolute_Coefficient"] = (
    coefficient_df["Coefficient"].abs()
)

coefficient_df["Odds_Ratio"] = np.exp(
    coefficient_df["Coefficient"].clip(-20, 20)
)

coefficient_df["Effect_Direction"] = np.where(
    coefficient_df["Coefficient"] >= 0,
    "RISK_INCREASING",
    "RISK_DECREASING",
)

display(
    coefficient_df
    .sort_values(
        "Absolute_Coefficient",
        ascending=False,
    )
    .head(25)
)


In [ ]:
top_coefficients = (
    coefficient_df
    .sort_values(
        "Absolute_Coefficient",
        ascending=False,
    )
    .head(15)
    .sort_values("Coefficient")
)

plt.figure(figsize=(10, 7))
plt.barh(
    top_coefficients["Feature"],
    top_coefficients["Coefficient"],
)
plt.title("Largest Logistic Regression Effects")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "07_top_coefficients.png",
    dpi=150,
)
plt.show()


## 11. False negative ve false positive analizi

In [ ]:
test_evaluation = df_training.loc[
    test_mask,
    [
        "Asset_ID",
        "Timestamp",
        "Operating_Mode",
        "Future_Event",
        "Vib_Radial_mm_s",
        "Vib_Tangential_mm_s",
        "Vib_Axial_mm_s",
        "Acc_RMS_Tangential_g",
        "Pk_Pk_Tangential_g",
        "Skin_Temp_C",
        "Output_Power_kW",
        "Deviation_Vib_Tangential",
        "Avg_3_Vib_Tangential",
    ],
].copy()

test_evaluation["Risk_Probability"] = test_probability
test_evaluation["Predicted_Future_Event"] = (
    test_prediction_selected
)

false_negatives = test_evaluation[
    (test_evaluation["Future_Event"] == 1)
    & (
        test_evaluation["Predicted_Future_Event"]
        == 0
    )
].sort_values("Risk_Probability", ascending=False)

false_positives = test_evaluation[
    (test_evaluation["Future_Event"] == 0)
    & (
        test_evaluation["Predicted_Future_Event"]
        == 1
    )
].sort_values("Risk_Probability", ascending=False)

print("False negatives:", len(false_negatives))
print("False positives:", len(false_positives))

display(false_negatives.head(20))
display(false_positives.head(20))


In [ ]:
timeline = test_evaluation.sort_values("Timestamp")

plt.figure(figsize=(14, 5))
plt.plot(
    timeline["Timestamp"],
    timeline["Risk_Probability"],
)
plt.axhline(
    selected_threshold,
    linestyle="--",
    label="Selected threshold",
)

event_rows = timeline[
    timeline["Future_Event"] == 1
]

plt.scatter(
    event_rows["Timestamp"],
    event_rows["Risk_Probability"],
    label="Observed Future Event",
)

plt.title("Test Period Risk Probability")
plt.xlabel("Timestamp")
plt.ylabel("Risk probability")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "08_test_risk_timeline.png",
    dpi=150,
)
plt.show()


## 12. Final modeli TRAIN + VALIDATION üzerinde eğitme

In [ ]:
development_mask = df_training["Data_Split"].isin(
    ["TRAIN", "VALIDATION"]
)

X_development = df_training.loc[
    development_mask,
    feature_columns,
].copy()

y_development = df_training.loc[
    development_mask,
    target_column,
].copy()

final_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42,
        ),
    ),
])

final_model.fit(
    X_development,
    y_development,
)

print(
    "Final model TRAIN + VALIDATION üzerinde eğitildi:",
    X_development.shape,
)


## 13. Tüm feature tablosunu skorlama

In [ ]:
X_scoring = df_features[feature_columns].copy()

risk_probability = final_model.predict_proba(
    X_scoring
)[:, 1]

predicted_event = (
    risk_probability >= selected_threshold
).astype(int)

output_columns = [
    "Asset_ID",
    "Timestamp",
    "Operating_Mode",
    "Speed_rpm",
    "Frequency_Hz",
    "Output_Power_kW",
    "Skin_Temp_C",
    "Vib_Radial_mm_s",
    "Vib_Tangential_mm_s",
    "Vib_Axial_mm_s",
    "Acc_RMS_Tangential_g",
    "Pk_Pk_Tangential_g",
    "Deviation_Vib_Radial",
    "Deviation_Vib_Tangential",
    "Deviation_Vib_Axial",
    "Avg_3_Vib_Radial",
    "Avg_3_Vib_Tangential",
    "Avg_3_Vib_Axial",
]

prediction_output = df_features[
    output_columns
].copy()

prediction_output["Risk_Probability"] = risk_probability
prediction_output["Predicted_Future_Event"] = predicted_event
prediction_output["Alarm_Threshold"] = selected_threshold
prediction_output["Alarm_Flag"] = (
    prediction_output["Risk_Probability"]
    >= selected_threshold
)

prediction_output["Condition_Score"] = (
    100
    * (
        1
        - prediction_output["Risk_Probability"]
    )
).clip(0, 100)

medium_threshold = selected_threshold / 2

prediction_output["Risk_Level"] = np.select(
    [
        prediction_output["Risk_Probability"]
        >= selected_threshold,
        prediction_output["Risk_Probability"]
        >= medium_threshold,
    ],
    ["HIGH", "MEDIUM"],
    default="LOW",
)

prediction_output["Model_Name"] = MODEL_NAME
prediction_output["Prediction_Timestamp"] = (
    pd.Timestamp.now(tz="UTC")
)

label_lookup = df_training[
    [
        "Asset_ID",
        "Timestamp",
        "Future_Event",
        "Data_Split",
    ]
].rename(
    columns={
        "Future_Event": "Actual_Future_Event",
    }
)

prediction_output = prediction_output.merge(
    label_lookup,
    on=["Asset_ID", "Timestamp"],
    how="left",
)

prediction_output["Actual_Future_Event"] = (
    prediction_output["Actual_Future_Event"]
    .astype("Int64")
)

display(
    prediction_output[
        [
            "Timestamp",
            "Operating_Mode",
            "Risk_Probability",
            "Condition_Score",
            "Risk_Level",
            "Alarm_Flag",
            "Actual_Future_Event",
            "Data_Split",
        ]
    ].tail(20)
)


In [ ]:
risk_level_counts = (
    prediction_output["Risk_Level"]
    .value_counts()
    .reindex(
        ["LOW", "MEDIUM", "HIGH"],
        fill_value=0,
    )
)

plt.figure(figsize=(7, 5))
plt.bar(
    risk_level_counts.index,
    risk_level_counts.values,
)
plt.title("Risk Level Distribution")
plt.xlabel("Risk level")
plt.ylabel("Row count")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "09_risk_level_distribution.png",
    dpi=150,
)
plt.show()


## 14. Model ve rapor dosyalarını kaydetme

In [ ]:
model_path = MODEL_DIR / f"{MODEL_NAME}.joblib"
metadata_path = MODEL_DIR / f"{MODEL_NAME}_metadata.json"
metrics_path = REPORT_DIR / f"{MODEL_NAME}_test_metrics.csv"
coefficients_path = REPORT_DIR / f"{MODEL_NAME}_coefficients.csv"
predictions_path = REPORT_DIR / f"{MODEL_NAME}_predictions.csv"

joblib.dump(final_model, model_path)

metadata = {
    "model_name": MODEL_NAME,
    "project_id": GCP_PROJECT_ID,
    "training_table": TRAINING_TABLE,
    "feature_table": FEATURE_TABLE,
    "selected_threshold": selected_threshold,
    "medium_threshold": medium_threshold,
    "target_definition": (
        "Abnormal sensor behavior in at least one "
        "of the next three active measurements."
    ),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "train_rows": len(X_train),
    "validation_rows": len(X_validation),
    "test_rows": len(X_test),
    "development_rows": len(X_development),
    "created_at_utc": str(pd.Timestamp.now(tz="UTC")),
}

with metadata_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

test_metrics.to_csv(metrics_path, index=False)
coefficient_df.to_csv(coefficients_path, index=False)
prediction_output.to_csv(predictions_path, index=False)

print("Model:", model_path)
print("Metadata:", metadata_path)
print("Metrics:", metrics_path)
print("Coefficients:", coefficients_path)
print("Predictions:", predictions_path)


## 15. BigQuery `pm_ml` tablolarına yazma

In [ ]:
dataset = bigquery.Dataset(
    f"{GCP_PROJECT_ID}.{ML_DATASET}"
)
dataset.location = BQ_LOCATION

client.create_dataset(
    dataset,
    exists_ok=True,
)

prediction_output["Timestamp"] = pd.to_datetime(
    prediction_output["Timestamp"],
    utc=True,
)
prediction_output["Prediction_Timestamp"] = pd.to_datetime(
    prediction_output["Prediction_Timestamp"],
    utc=True,
)

metrics_output = test_metrics.copy()
metrics_output["Model_Name"] = MODEL_NAME
metrics_output["Evaluation_Split"] = "TEST"
metrics_output["Evaluation_Timestamp"] = (
    pd.Timestamp.now(tz="UTC")
)

coefficient_output = coefficient_df.copy()
coefficient_output["Model_Name"] = MODEL_NAME
coefficient_output["Training_Scope"] = (
    "TRAIN_PLUS_VALIDATION"
)
coefficient_output["Generated_Timestamp"] = (
    pd.Timestamp.now(tz="UTC")
)

job_config = bigquery.LoadJobConfig(
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

jobs = [
    client.load_table_from_dataframe(
        prediction_output,
        PREDICTION_TABLE,
        job_config=job_config,
    ),
    client.load_table_from_dataframe(
        metrics_output,
        METRICS_TABLE,
        job_config=job_config,
    ),
    client.load_table_from_dataframe(
        coefficient_output,
        COEFFICIENT_TABLE,
        job_config=job_config,
    ),
]

for job in jobs:
    job.result()

print("BigQuery tabloları yazıldı:")
print(PREDICTION_TABLE)
print(METRICS_TABLE)
print(COEFFICIENT_TABLE)


## 16. BigQuery yükleme kontrolü

In [ ]:
check_query = f'''
SELECT
    COUNT(*) AS prediction_rows,
    MIN(Timestamp) AS first_prediction,
    MAX(Timestamp) AS last_prediction,
    AVG(Risk_Probability) AS average_risk,
    AVG(Condition_Score) AS average_condition_score,
    COUNTIF(Risk_Level = 'HIGH') AS high_risk_rows,
    COUNTIF(Alarm_Flag) AS alarm_rows
FROM `{PREDICTION_TABLE}`
'''

display(
    client.query(
        check_query
    ).to_dataframe()
)


## 17. Otomatik model özeti

In [ ]:
selected_metrics = test_metrics[
    test_metrics["Model"]
    == "Logistic Regression — selected threshold"
].iloc[0]

summary = pd.DataFrame({
    "Item": [
        "Model",
        "Selected threshold",
        "Test ROC-AUC",
        "Test average precision",
        "Test precision",
        "Test recall",
        "Test F1",
        "Test F2",
        "False negatives",
        "False positives",
        "Scored rows",
        "High-risk rows",
        "BigQuery prediction table",
    ],
    "Value": [
        MODEL_NAME,
        round(selected_threshold, 4),
        round(selected_metrics["ROC_AUC"], 4),
        round(selected_metrics["Average_Precision"], 4),
        round(selected_metrics["Precision"], 4),
        round(selected_metrics["Recall"], 4),
        round(selected_metrics["F1"], 4),
        round(selected_metrics["F2"], 4),
        len(false_negatives),
        len(false_positives),
        len(prediction_output),
        int(
            (
                prediction_output["Risk_Level"]
                == "HIGH"
            ).sum()
        ),
        PREDICTION_TABLE,
    ],
})

display(summary)


## Sonuçların yorumu

- `Risk_Probability`: Sonraki üç aktif ölçüm içinde anormal sensör davranışı görülme olasılığı.
- `Condition_Score`: `100 × (1 − Risk_Probability)`.
- `HIGH`: risk seçilen alarm threshold'una eşit veya yüksek.
- `MEDIUM`: risk threshold'un yarısı ile threshold arasında.
- `LOW`: risk threshold'un yarısından düşük.

## Sınırlamalar

- Tek motor ve sınırlı tarih aralığı kullanılmıştır.
- Gerçek bakım kaydı veya doğrulanmış arıza etiketi yoktur.
- Hedef sensör eşiklerinden türetilmiş proxy etikettir.
- Condition score gerçek fiziksel sağlık yüzdesi veya Remaining Useful Life değildir.
- Model farklı motorlarda doğrulanmadan genellenmemelidir.

## Power BI tabloları

- `pm_clean.motor_measurements`
- `pm_features.motor_condition_features`
- `pm_ml.predictions`
- `pm_ml.model_metrics`
- `pm_ml.model_coefficients`
